# Final warming date on 1-50 hPa pressure levels

This notebook computes yearly final warming dates (FWD) for BWCN, B2000WCN001002, B2000WCN_NOCOUPL001002, and B2000WCN007009010011_Clim3D.

Workflow:
- read annual `U` files and the matching surface pressure (`PS`) field;
- reconstruct CAM hybrid mid-level pressure from `hyam * P0 + hybm * PS`;
- interpolate `U` to a compact 1-50 hPa grid chosen from representative hybrid levels, using linear interpolation in log-pressure;
- take zonal-mean `U` at 60N;
- find the first Jan-Jun reversal following the Gerber & Butler style rule used in `code/find_FW.py`.

The timefixed inputs may contain whole-day NaNs inserted by `Data_nan_fill.ipynb`. By default, a year/level with any Jan-Jun NaN is returned as NaN instead of being silently treated as a valid final warming date.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from pathlib import Path
import gc
import re
import warnings
from contextlib import nullcontext

import numpy as np
import pandas as pd
import xarray as xr

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else nullcontext()

# ============================================================
# Configuration
# ============================================================

BASE_DIR = Path('/mnt/soclim0/public_data/weiji')

CASES = {
    'B2000WCN007009010011_Clim3D': {
        'root': BASE_DIR / 'B2000WCN007009010011_Clim3D_timefixed',
    },
    'B2000WCN_NOCOUPL001002': {
        'root': BASE_DIR / 'B2000WCN_NOCOUPL001002_timefixed',
    },
    'B2000WCN001002': {
        'root': BASE_DIR / 'B2000WCN001002_timefixed',
    },
    'BWCN': {
        'root': BASE_DIR / 'BWCN',
    },
}

# Representative hybrid mid-level pressures near 60N from
# B2000WCN001002_timefixed/U/B2000WCN.sample.cam.h3.0001.U.nc are roughly:
# 1.24, 1.61, 2.08, 2.67, 3.40, 4.32, 5.47, 6.87, 8.60,
# 10.71, 13.26, 16.35, 20.06, 24.48, 29.73, 35.92, 43.19 hPa.
# Round nearby hybrid levels to integer hPa and force 1, 5, 10, 50 hPa.
PLEV_FWD_HPA = np.array([1, 2, 3, 4, 5, 7, 9, 10, 11, 13, 16, 20, 24, 30, 36, 43, 50], dtype=np.float64)
PLEV_FWD_PA = PLEV_FWD_HPA * 100.0

TARGET_LAT = 60.0
WESTERLY_RUN_LENGTH = 10
EXPECTED_JAN_JUN_DAYS = 181
PURE_PRESSURE_HYBM_TOL = 1.0e-12
MONTH_LENGTHS_NOLEAP = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], dtype=np.int16)
MONTH_ENDS_NOLEAP = np.cumsum(MONTH_LENGTHS_NOLEAP)

# If True, a year/level with any Jan-Jun NaN is marked as missing.
# This avoids false FWD detections across filled all-NaN days.
REQUIRE_COMPLETE_JAN_JUN = True

DEBUG_MODE = False
DEBUG_CASES = ['B2000WCN001002']
DEBUG_YEARS = [1, 2]

OVERWRITE = False
OUTPUT_SUBDIR = 'final_warming_date_debug' if DEBUG_MODE else 'final_warming_date'
CASES_TO_RUN = DEBUG_CASES if DEBUG_MODE else list(CASES)
YEARS_TO_RUN = DEBUG_YEARS if DEBUG_MODE else None

NETCDF_ENGINE = 'netcdf4'
COMPLEVEL = 1
FILL_INT = np.int32(-9999)
YEAR_RE = re.compile(r'\.h3\.(\d{4})\.')

print('DEBUG_MODE =', DEBUG_MODE)
print('OUTPUT_SUBDIR =', OUTPUT_SUBDIR)
print('CASES_TO_RUN =', CASES_TO_RUN)
print('YEARS_TO_RUN =', YEARS_TO_RUN)
print('TARGET_LAT =', TARGET_LAT)
print('REQUIRE_COMPLETE_JAN_JUN =', REQUIRE_COMPLETE_JAN_JUN)


In [ ]:
# ============================================================
# File discovery and calendar helpers
# ============================================================

def progress_iter(iterable, **kwargs):
    return tqdm(iterable, **kwargs)


def parse_year(path):
    m = YEAR_RE.search(Path(path).name)
    if m is None:
        return None
    return int(m.group(1))


def discover_var_files(case_root, var, years=None):
    files = sorted((Path(case_root) / var).glob(f'*.{var}.nc'), key=lambda p: (parse_year(p) or 10**9, p.name))
    if years is not None:
        year_set = {int(y) for y in years}
        files = [p for p in files if parse_year(p) in year_set]
    return files


def discover_year_pairs(case_root, years=None):
    u_by_year = {parse_year(p): p for p in discover_var_files(case_root, 'U', years=years) if parse_year(p) is not None}
    ps_by_year = {parse_year(p): p for p in discover_var_files(case_root, 'PS', years=years) if parse_year(p) is not None}

    u_years = sorted(u_by_year)
    missing_ps = sorted(set(u_by_year) - set(ps_by_year))
    missing_u = sorted(set(ps_by_year) - set(u_by_year))
    pairs = [(year, u_by_year[year], ps_by_year.get(year)) for year in u_years]
    return pairs, missing_u, missing_ps


def date_to_month_doy(date_values):
    date_values = np.asarray(date_values, dtype=np.int64)
    mmdd = date_values % 10000
    month = (mmdd // 100).astype(np.int16)
    day = (mmdd % 100).astype(np.int16)
    doy = np.empty(date_values.shape, dtype=np.int16)
    for m in range(1, 13):
        mask = month == m
        if np.any(mask):
            doy[mask] = int(MONTH_ENDS_NOLEAP[m - 2]) + day[mask] if m > 1 else day[mask]
    return month, day, doy


def doy_to_month_day(doy_values):
    doy_values = np.asarray(doy_values, dtype=np.int16)
    month = np.searchsorted(MONTH_ENDS_NOLEAP, doy_values, side='left').astype(np.int16) + 1
    prev_end = np.where(month > 1, MONTH_ENDS_NOLEAP[month - 2], 0).astype(np.int16)
    day = (doy_values - prev_end).astype(np.int16)
    return month, day


def ymd_from_year_doy(year, doy_values):
    month, day = doy_to_month_day(doy_values)
    return (int(year) * 10000 + month.astype(np.int32) * 100 + day.astype(np.int32)).astype(np.int32)


def calendar_for_annual_file(ds, fallback_year):
    ntime = int(ds.sizes.get('time', 0))
    if 'date' in ds:
        dates = ds['date'].values.astype(np.int64)
        month, day, doy = date_to_month_doy(dates)
        model_year = (dates // 10000).astype(np.int32)
        return {
            'year': model_year,
            'month': month,
            'day': day,
            'doy': doy,
            'date': dates.astype(np.int32),
        }

    doy = np.arange(1, ntime + 1, dtype=np.int16)
    month, day = doy_to_month_day(doy)
    return {
        'year': np.full(ntime, int(fallback_year), dtype=np.int32),
        'month': month,
        'day': day,
        'doy': doy,
        'date': ymd_from_year_doy(fallback_year, doy),
    }


def threshold_for_pressure(plev_pa):
    # find_FW.py uses 0 m/s for levels <= 10 hPa and 7 m/s below that.
    plev_pa = np.asarray(plev_pa, dtype=np.float64)
    return np.where(plev_pa <= 1000.0, 0.0, 7.0).astype(np.float32)


In [ ]:
# ============================================================
# Hybrid-to-pressure interpolation and FWD algorithm
# ============================================================

def _open_dataset(path):
    try:
        return xr.open_dataset(path, decode_times=False, engine=NETCDF_ENGINE)
    except Exception:
        return xr.open_dataset(path, decode_times=False)


def interp_logp_numpy(v_hyb, p_hyb, p_tgt_pa):
    v = np.asarray(v_hyb, dtype=np.float64)
    p = np.asarray(p_hyb, dtype=np.float64)
    p_tgt_pa = np.asarray(p_tgt_pa, dtype=np.float64)

    if v.shape != p.shape:
        raise ValueError(f'v_hyb and p_hyb shape mismatch: {v.shape} vs {p.shape}')
    if v.shape[-1] < 2:
        return np.full(v.shape[:-1] + (len(p_tgt_pa),), np.nan, dtype=np.float32)

    with np.errstate(invalid='ignore', divide='ignore'):
        log_p = np.where((p > 0.0) & np.isfinite(p) & np.isfinite(v), np.log(p), np.nan)

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        first = np.nanmedian(log_p[..., 0])
        last = np.nanmedian(log_p[..., -1])

    if np.isfinite(first) and np.isfinite(last) and first > last:
        log_p = log_p[..., ::-1]
        v = v[..., ::-1]

    out = np.full(v.shape[:-1] + (len(p_tgt_pa),), np.nan, dtype=np.float64)
    left_p = log_p[..., :-1]
    right_p = log_p[..., 1:]
    left_v = v[..., :-1]
    right_v = v[..., 1:]
    log_targets = np.log(p_tgt_pa)

    for j, target in enumerate(log_targets):
        bracket = (left_p <= target) & (target <= right_p)
        has_bracket = bracket.any(axis=-1)
        k = bracket.argmax(axis=-1)
        take_k = np.expand_dims(k, axis=-1)

        p0 = np.take_along_axis(left_p, take_k, axis=-1)[..., 0]
        p1 = np.take_along_axis(right_p, take_k, axis=-1)[..., 0]
        v0 = np.take_along_axis(left_v, take_k, axis=-1)[..., 0]
        v1 = np.take_along_axis(right_v, take_k, axis=-1)[..., 0]

        with np.errstate(invalid='ignore', divide='ignore'):
            weight = (target - p0) / (p1 - p0)
            y = v0 + weight * (v1 - v0)

        good = has_bracket & np.isfinite(y) & np.isfinite(weight)
        out[..., j] = np.where(good, y, np.nan)

    return out.astype(np.float32, copy=False)


def pure_pressure_mask_for_targets(hyam, hybm, p0, p_tgt_pa):
    p_1d = np.asarray(hyam, dtype=np.float64) * float(p0)
    hybm = np.asarray(hybm, dtype=np.float64)
    pure_mask = np.abs(hybm) <= PURE_PRESSURE_HYBM_TOL
    if pure_mask.sum() < 2:
        return None, None

    p_pure = p_1d[pure_mask]
    if np.nanmin(p_tgt_pa) < np.nanmin(p_pure) or np.nanmax(p_tgt_pa) > np.nanmax(p_pure):
        return None, None

    return pure_mask, p_pure


def lat_bracket_indices(lat_values, target_lat):
    lat_values = np.asarray(lat_values, dtype=np.float64)
    order = np.argsort(lat_values)
    lat_sorted = lat_values[order]

    exact = np.where(np.isclose(lat_sorted, target_lat, rtol=0.0, atol=1.0e-10))[0]
    if exact.size:
        return [int(order[exact[0]])], 0.0

    pos = int(np.searchsorted(lat_sorted, target_lat, side='left'))
    if pos <= 0 or pos >= len(lat_sorted):
        raise ValueError(f'TARGET_LAT={target_lat} is outside source latitude range {lat_sorted[0]}..{lat_sorted[-1]}')

    lower = pos - 1
    upper = pos
    lat0 = lat_sorted[lower]
    lat1 = lat_sorted[upper]
    weight = float((target_lat - lat0) / (lat1 - lat0))
    return [int(order[lower]), int(order[upper])], weight


def collapse_selected_lat(arr, weight):
    # Input arrays have latitude on axis=1 and contain either one exact-lat
    # slice or the two bracketing latitude slices returned above.
    if arr.shape[1] == 1:
        return arr[:, 0, ...]
    return (1.0 - weight) * arr[:, 0, ...] + weight * arr[:, 1, ...]


def load_u60_plev_zonal_mean(year, u_file, ps_file):
    ds_u = _open_dataset(u_file)
    ds_ps = None
    try:
        required = {'U', 'hyam', 'hybm'}
        missing = required - set(ds_u.variables)
        if missing:
            raise ValueError(f'{u_file.name} missing required variables: {sorted(missing)}')

        if 'lat' not in ds_u['U'].dims:
            raise ValueError(f'{u_file.name}: U has no lat dimension')

        p0 = float(ds_u['P0'].values) if 'P0' in ds_u else 100000.0
        hyam = ds_u['hyam'].values.astype(np.float64)
        hybm = ds_u['hybm'].values.astype(np.float64)
        lat_indices, lat_weight = lat_bracket_indices(ds_u['lat'].values, TARGET_LAT)
        pure_mask, p_pure = pure_pressure_mask_for_targets(hyam, hybm, p0, PLEV_FWD_PA)

        if pure_mask is not None:
            lev_index = np.where(pure_mask)[0]
            u_sel = ds_u['U'].isel(lat=lat_indices, lev=lev_index)
            if 'lon' in u_sel.dims:
                u_lat = u_sel.transpose('time', 'lat', 'lon', 'lev').values
                u60 = collapse_selected_lat(u_lat, lat_weight)
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore', category=RuntimeWarning)
                    u_hyb_zm = np.nanmean(u60, axis=1)
            else:
                u_lat = u_sel.transpose('time', 'lat', 'lev').values
                u_hyb_zm = collapse_selected_lat(u_lat, lat_weight)
            p_hyb = np.broadcast_to(p_pure[None, :], u_hyb_zm.shape)
            u_plev_zm = interp_logp_numpy(u_hyb_zm, p_hyb, PLEV_FWD_PA)
            calendar = calendar_for_annual_file(ds_u, fallback_year=year)
            return u_plev_zm.astype(np.float32), calendar, {'pressure_path': 'pure_pressure_hybm0'}

        if 'PS' in ds_u:
            ps = ds_u['PS']
        else:
            if ps_file is None:
                raise ValueError(f'{u_file.name} has no embedded PS and no paired PS file was found')
            ds_ps = _open_dataset(ps_file)
            if 'PS' not in ds_ps:
                raise ValueError(f'{ps_file.name} missing PS')
            ps = ds_ps['PS']

        u_sel = ds_u['U'].isel(lat=lat_indices)
        ps_sel = ps.isel(lat=lat_indices)

        if 'lon' in u_sel.dims:
            u_lat = u_sel.transpose('time', 'lat', 'lon', 'lev').values
            u_arr = collapse_selected_lat(u_lat, lat_weight)
        else:
            u_lat = u_sel.transpose('time', 'lat', 'lev').values
            u_arr = collapse_selected_lat(u_lat, lat_weight)[:, None, :]

        if 'lon' in ps_sel.dims:
            ps_lat = ps_sel.transpose('time', 'lat', 'lon').values.astype(np.float64)
            ps_arr = collapse_selected_lat(ps_lat, lat_weight)
        else:
            ps_lat = ps_sel.transpose('time', 'lat').values.astype(np.float64)
            ps_arr = collapse_selected_lat(ps_lat, lat_weight)[:, None]

        p_mid = hyam[None, None, :] * p0 + hybm[None, None, :] * ps_arr[:, :, None]
        u_plev = interp_logp_numpy(u_arr, p_mid, PLEV_FWD_PA)

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', category=RuntimeWarning)
            u_plev_zm = np.nanmean(u_plev, axis=1).astype(np.float32)

        calendar = calendar_for_annual_file(ds_u, fallback_year=year)
        return u_plev_zm, calendar, {'pressure_path': 'full_hybrid_ps'}
    finally:
        ds_u.close()
        if ds_ps is not None:
            ds_ps.close()


def has_westerly_return(values, start_index, thresh):
    above = np.isfinite(values) & (values > thresh)
    last_start = len(values) - WESTERLY_RUN_LENGTH
    for i in range(start_index + 1, last_start + 1):
        if above[i:i + WESTERLY_RUN_LENGTH].all():
            return True
    return False


def find_final_warming_one_level(values, doys, months, days, dates, plev_pa):
    values = np.asarray(values, dtype=np.float64)
    finite = np.isfinite(values)
    thresh = float(threshold_for_pressure([plev_pa])[0])

    if REQUIRE_COMPLETE_JAN_JUN and (len(values) != EXPECTED_JAN_JUN_DAYS or not finite.all()):
        return np.nan, np.nan, FILL_INT, FILL_INT, FILL_INT, thresh

    for i, val in enumerate(values):
        if not np.isfinite(val):
            continue
        if val < thresh and not has_westerly_return(values, i, thresh):
            return float(doys[i]), float(i), np.int32(dates[i]), np.int32(months[i]), np.int32(days[i]), thresh

    return np.nan, np.nan, FILL_INT, FILL_INT, FILL_INT, thresh


def compute_year_fwd(year, u_file, ps_file):
    u_plev_zm, calendar, meta = load_u60_plev_zonal_mean(year, u_file, ps_file)
    month = calendar['month']
    jan_jun = (month >= 1) & (month <= 6)

    values = u_plev_zm[jan_jun, :]
    doys = calendar['doy'][jan_jun]
    months = calendar['month'][jan_jun]
    days = calendar['day'][jan_jun]
    dates = calendar['date'][jan_jun]

    nlev = len(PLEV_FWD_PA)
    fwd_doy = np.full(nlev, np.nan, dtype=np.float32)
    fwd_index0 = np.full(nlev, np.nan, dtype=np.float32)
    fwd_date = np.full(nlev, FILL_INT, dtype=np.int32)
    fwd_month = np.full(nlev, FILL_INT, dtype=np.int32)
    fwd_day = np.full(nlev, FILL_INT, dtype=np.int32)
    n_valid = np.zeros(nlev, dtype=np.int16)
    has_nan = np.zeros(nlev, dtype=np.int8)
    thresholds = threshold_for_pressure(PLEV_FWD_PA)

    for k, plev in enumerate(PLEV_FWD_PA):
        series = values[:, k]
        n_valid[k] = np.int16(np.isfinite(series).sum())
        has_nan[k] = np.int8(not np.isfinite(series).all() or len(series) != EXPECTED_JAN_JUN_DAYS)
        doy, idx0, date_int, mm, dd, _ = find_final_warming_one_level(
            series, doys, months, days, dates, plev
        )
        fwd_doy[k] = doy
        fwd_index0[k] = idx0
        fwd_date[k] = date_int
        fwd_month[k] = mm
        fwd_day[k] = dd

    return {
        'FWD_dayofyear': fwd_doy,
        'FWD_day_index_janjun0': fwd_index0,
        'FWD_date': fwd_date,
        'FWD_month': fwd_month,
        'FWD_day_of_month': fwd_day,
        'n_valid_janjun': n_valid,
        'has_nan_janjun': has_nan,
        'threshold': thresholds,
        'n_janjun_days': int(values.shape[0]),
        'pressure_path': meta.get('pressure_path', ''),
    }


In [ ]:
# ============================================================
# Dataset writing and case driver
# ============================================================

def fwd_output_file(case_root, case_name):
    return Path(case_root) / OUTPUT_SUBDIR / f'{case_name}_FWD_plev_year.nc'


def fwd_summary_file(case_root, case_name):
    return Path(case_root) / OUTPUT_SUBDIR / f'{case_name}_FWD_processing_summary.csv'


def output_encoding(ds):
    encoding = {}
    for name, da in ds.data_vars.items():
        enc = {'zlib': True, 'complevel': COMPLEVEL}
        if np.issubdtype(da.dtype, np.floating):
            enc['_FillValue'] = np.nan
        elif name in {'FWD_date', 'FWD_month', 'FWD_day_of_month'}:
            enc['_FillValue'] = FILL_INT
        encoding[name] = enc
    return encoding


def save_dataset_atomic(ds, out_file):
    out_file = Path(out_file)
    out_file.parent.mkdir(parents=True, exist_ok=True)
    if out_file.exists() and out_file.stat().st_size > 0 and not OVERWRITE:
        print(f'[SKIP] existing: {out_file}')
        return out_file

    tmp = out_file.with_name(out_file.name + '.tmp')
    if tmp.exists():
        tmp.unlink()

    print(f'[WRITE] {out_file}')
    ds.to_netcdf(
        tmp,
        engine=NETCDF_ENGINE,
        format='NETCDF4',
        encoding=output_encoding(ds),
    )
    tmp.replace(out_file)
    return out_file


def build_case_dataset(case_name, case_root, years, arrays, records):
    coords = {
        'year': np.asarray(years, dtype=np.int32),
        'plev': PLEV_FWD_PA.astype(np.float64),
        'plev_hpa': ('plev', PLEV_FWD_HPA.astype(np.float32)),
    }

    ds = xr.Dataset(
        {
            'FWD_dayofyear': (('year', 'plev'), arrays['FWD_dayofyear'].astype(np.float32)),
            'FWD_day_index_janjun0': (('year', 'plev'), arrays['FWD_day_index_janjun0'].astype(np.float32)),
            'FWD_date': (('year', 'plev'), arrays['FWD_date'].astype(np.int32)),
            'FWD_month': (('year', 'plev'), arrays['FWD_month'].astype(np.int32)),
            'FWD_day_of_month': (('year', 'plev'), arrays['FWD_day_of_month'].astype(np.int32)),
            'n_valid_janjun': (('year', 'plev'), arrays['n_valid_janjun'].astype(np.int16)),
            'has_nan_janjun': (('year', 'plev'), arrays['has_nan_janjun'].astype(np.int8)),
            'threshold_u': (('plev',), threshold_for_pressure(PLEV_FWD_PA).astype(np.float32)),
        },
        coords=coords,
    )

    ds['plev'].attrs.update({'long_name': 'pressure', 'units': 'Pa', 'positive': 'down'})
    ds['plev_hpa'].attrs.update({'long_name': 'pressure', 'units': 'hPa', 'positive': 'down'})
    ds['year'].attrs.update({'long_name': 'model year parsed from source filename'})
    ds['FWD_dayofyear'].attrs.update({'long_name': 'final warming day of year', 'units': 'day', 'missing_value': 'NaN'})
    ds['FWD_day_index_janjun0'].attrs.update({
        'long_name': '0-based Jan-Jun day index of final warming',
        'description': 'This matches the 0-based convention returned by code/find_FW.py.',
        'missing_value': 'NaN',
    })
    ds['FWD_date'].attrs.update({'long_name': 'final warming date as YYYYMMDD integer', 'missing_value': int(FILL_INT)})
    ds['FWD_month'].attrs.update({'long_name': 'final warming calendar month', 'missing_value': int(FILL_INT)})
    ds['FWD_day_of_month'].attrs.update({'long_name': 'final warming calendar day of month', 'missing_value': int(FILL_INT)})
    ds['n_valid_janjun'].attrs.update({'long_name': 'number of finite Jan-Jun U values used at this level'})
    ds['has_nan_janjun'].attrs.update({'long_name': '1 if this year/level had any Jan-Jun NaN or incomplete Jan-Jun days'})
    ds['threshold_u'].attrs.update({'long_name': 'wind reversal threshold', 'units': 'm s-1'})

    ok_records = [r for r in records if r.get('status') == 'ok']
    ds.attrs.update({
        'title': 'Final warming date by model year and pressure level',
        'case_name': case_name,
        'case_root': str(case_root),
        'source_variable': 'U',
        'target_latitude_degrees_north': float(TARGET_LAT),
        'zonal_mean': 'mean over lon after manual two-point latitude interpolation to 60N and compact 1-50 hPa pressure levels',
        'pressure_interpolation': 'linear interpolation in log(pressure); pure hybm=0 pressure levels are used directly when all target levels are bracketed there, otherwise CAM hybrid pressure hyam*P0 + hybm*PS is used',
        'pressure_units': 'Pa',
        'plev_values_hpa': ','.join(str(int(p)) for p in PLEV_FWD_HPA),
        'plev_values_pa': ','.join(str(int(p)) for p in PLEV_FWD_PA),
        'algorithm': f'first Jan-Jun day with U below threshold and no later {WESTERLY_RUN_LENGTH}-day continuous westerly return before July',
        'threshold_definition': '0 m/s for p <= 1000 Pa (<=10 hPa); 7 m/s for p > 1000 Pa',
        'nan_policy': 'If REQUIRE_COMPLETE_JAN_JUN is True, any Jan-Jun NaN or incomplete Jan-Jun days returns NaN for that year/level.',
        'require_complete_jan_jun': str(REQUIRE_COMPLETE_JAN_JUN),
        'source_file_count_ok': len(ok_records),
        'debug_mode': str(DEBUG_MODE),
    })
    return ds


def initialize_output_arrays(nyear, nlev):
    return {
        'FWD_dayofyear': np.full((nyear, nlev), np.nan, dtype=np.float32),
        'FWD_day_index_janjun0': np.full((nyear, nlev), np.nan, dtype=np.float32),
        'FWD_date': np.full((nyear, nlev), FILL_INT, dtype=np.int32),
        'FWD_month': np.full((nyear, nlev), FILL_INT, dtype=np.int32),
        'FWD_day_of_month': np.full((nyear, nlev), FILL_INT, dtype=np.int32),
        'n_valid_janjun': np.zeros((nyear, nlev), dtype=np.int16),
        'has_nan_janjun': np.ones((nyear, nlev), dtype=np.int8),
    }


def process_case(case_name):
    cfg = CASES[case_name]
    root = Path(cfg['root'])
    out_file = fwd_output_file(root, case_name)
    summary_file = fwd_summary_file(root, case_name)

    print('\n' + '=' * 100)
    print(f'CASE: {case_name}')
    print(f'ROOT: {root}')
    print('=' * 100)

    if out_file.exists() and out_file.stat().st_size > 0 and not OVERWRITE:
        print(f'[SKIP] existing output: {out_file}')
        return out_file

    if not root.exists():
        raise FileNotFoundError(f'Missing case root: {root}')

    pairs, missing_u, missing_ps = discover_year_pairs(root, years=YEARS_TO_RUN)
    if missing_u:
        print(f'[WARN] years with PS but no U: {missing_u[:10]}')
    if missing_ps:
        print(f'[WARN] years with U but no PS: {missing_ps[:10]}')
    if not pairs:
        raise FileNotFoundError(f'No paired U/PS annual files found under {root}')

    years = [p[0] for p in pairs]
    print(f'years: n={len(years)}, first={years[0]}, last={years[-1]}')

    arrays = initialize_output_arrays(len(years), len(PLEV_FWD_PA))
    records = []

    for i, (year, u_file, ps_file) in enumerate(progress_iter(pairs, desc=f'{case_name} years', unit='year')):
        rec = {
            'case': case_name,
            'year': int(year),
            'u_file': str(u_file),
            'ps_file': str(ps_file) if ps_file is not None else '',
            'status': 'ok',
            'message': '',
        }
        try:
            result = compute_year_fwd(year, u_file, ps_file)
            for key in arrays:
                arrays[key][i, :] = result[key]
            rec['n_janjun_days'] = result['n_janjun_days']
            rec['n_levels_with_fwd'] = int(np.isfinite(result['FWD_dayofyear']).sum())
            rec['n_levels_with_nan_janjun'] = int(result['has_nan_janjun'].sum())
            rec['pressure_path'] = result.get('pressure_path', '')
        except Exception as exc:
            rec['status'] = 'error'
            rec['message'] = repr(exc)
            print(f'[ERROR] {case_name} year {year}: {exc!r}')
        records.append(rec)
        gc.collect()

    ds = build_case_dataset(case_name, root, years, arrays, records)
    written = save_dataset_atomic(ds, out_file)
    ds.close()

    summary_file.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(records).to_csv(summary_file, index=False)
    print(f'[WRITE] {summary_file}')
    return written


In [ ]:
# ============================================================
# Run all requested cases
# ============================================================

all_outputs = {}
for case_name in progress_iter(CASES_TO_RUN, desc='Cases', unit='case'):
    all_outputs[case_name] = process_case(case_name)

print('\nDone. Outputs:')
for case_name, out in all_outputs.items():
    print(case_name)
    print('  ', out)
